In [ ]:
import sys
sys.path.append("/path/to/example/autocompute/static")
from g16_env import load_gaussian_env
load_gaussian_env()

In [ ]:
import os
import subprocess
import re
import csv
from concurrent.futures import ThreadPoolExecutor
from openbabel import openbabel
from openpyxl import Workbook
import time

In [ ]:

MAX_TASKS = 2

In [ ]:

os.makedirs('dimer_gas/opt+freq/success', exist_ok=True)
os.makedirs('dimer_gas/opt+freq/failure', exist_ok=True)

In [ ]:

base_dir = 'dimer_gas/opt+freq'
success_dir = os.path.join(base_dir, 'success')
failure_dir = os.path.join(base_dir, 'failure')

In [ ]:

def run_gaussian_ion_gas_optfreq(file, success_dir, failure_dir):
    
    
    
    base_name = os.path.splitext(file)[0]
    output_file = base_name + '.out'
    chk_file = base_name + '.chk'
    
    
    try:
        subprocess.run(['g16', file, output_file], check=True)
        
        
        with open(output_file, 'r') as f:
            content = f.read()
            
            if 'Normal termination' in content:
                
                os.rename(file, os.path.join(success_dir, file))
                os.rename(output_file, os.path.join(success_dir, output_file))
                os.rename(chk_file, os.path.join(success_dir, chk_file))
                return output_file, True
            else:
                
                os.rename(file, os.path.join(failure_dir, file))
                os.rename(output_file, os.path.join(failure_dir, output_file))
                os.rename(chk_file, os.path.join(failure_dir, chk_file))
    except subprocess.CalledProcessError:
        
        os.rename(file, os.path.join(failure_dir, file))
        if os.path.exists(output_file):
            os.rename(output_file, os.path.join(failure_dir, output_file))
        if os.path.exists(chk_file):
            os.rename(chk_file, os.path.join(failure_dir, chk_file))

    return None, False

In [ ]:

def main():
    
    
    start_time = time.time() 
    
    
    gjf_files = [f for f in os.listdir('.') if os.path.isfile(f) and f.endswith('.gjf')]
    
    
    with ThreadPoolExecutor(max_workers=MAX_TASKS) as executor:
        
        futures = [executor.submit(run_gaussian_ion_gas_optfreq, file, success_dir, failure_dir) for file in gjf_files] 
        
        
        results = [f.result() for f in futures]
    
    
    success_results = [result for result in results if result[1]]
    
    
    end_time = time.time()
    
    
    elapsed_time = end_time - start_time
    print(f"The code ran for {elapsed_time} seconds.")



In [ ]:
if __name__ == '__main__':
    main()